In [3]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Added on_bad_lines='skip' and engine='python' to handle the malformed quoted row
talks = pd.read_csv(
    "/content/ted_talks_cleaned.csv",
    dtype={"talk_id": str},
    on_bad_lines='skip',
    engine='python'
)
interactions = pd.read_csv(
    "/content/user_interactions.csv",
    dtype={"user_id": str, "talk_id": str}
)

talks["talk_id"] = talks["talk_id"].astype(str)
interactions["talk_id"] = interactions["talk_id"].astype(str)
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce")
interactions = interactions.dropna(subset=["user_id", "talk_id", "rating"])

In [4]:
positive = interactions[interactions["rating"] >= 4].copy()
positive["row_order"] = positive.index

test_data = (
    positive.sort_values("row_order")
    .groupby("user_id")
    .tail(1)[["user_id", "talk_id"]]
)

train_data = interactions.merge(
    test_data.assign(is_test=True),
    on=["user_id", "talk_id"],
    how="left"
)

train_data = train_data[train_data["is_test"].isna()].drop(columns="is_test")

print("Users evaluated:", len(test_data))
print("Training interactions:", len(train_data))

Users evaluated: 30
Training interactions: 750


In [5]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2
)

talk_vectors = vectorizer.fit_transform(talks["combined_text"].fillna(""))

user_item = train_data.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean"
).fillna(0)

user_similarity = cosine_similarity(user_item)
user_similarity = pd.DataFrame(
    user_similarity,
    index=user_item.index,
    columns=user_item.index
)

print("Content and collaborative models are ready.")

Content and collaborative models are ready.


In [13]:
def get_content_recommendations(user_id, top_n=10):
    history = train_data[train_data["user_id"] == user_id]
    liked_ids = history.loc[history["rating"] >= 4, "talk_id"].tolist()

    liked_rows = talks.index[talks["talk_id"].isin(liked_ids)].tolist()
    if not liked_rows:
        return []

    # Calculate mean and convert from np.matrix to a standard np.array to prevent TypeError
    profile = np.asarray(talk_vectors[liked_rows].mean(axis=0))
    scores = cosine_similarity(profile, talk_vectors).flatten()

    result = pd.DataFrame({
        "talk_id": talks["talk_id"],
        "score": scores
    })

    result = result[~result["talk_id"].isin(history["talk_id"])]
    return result.sort_values("score", ascending=False)["talk_id"].head(top_n).tolist()


def get_collaborative_recommendations(user_id, top_n=10):
    if user_id not in user_item.index:
        return []

    # Drop the target user to find neighbors
    similar_users = user_similarity[user_id].drop(user_id)

    # Select ratings for other users to match similar_users
    other_users_ratings = user_item.drop(index=user_id)

    # Compute weighted scores using aligned Series/DataFrame objects
    weighted_scores = other_users_ratings.T.dot(similar_users)
    similarity_totals = other_users_ratings.ne(0).T.dot(similar_users)

    scores = weighted_scores / similarity_totals.replace(0, np.nan)
    scores = scores.dropna()

    rated_ids = train_data.loc[train_data["user_id"] == user_id, "talk_id"]
    scores = scores.drop(index=rated_ids, errors="ignore")

    return scores.sort_values(ascending=False).head(top_n).index.tolist()

In [17]:
def precision_recall_at_k(recommendations, relevant_talks, k=10):
    top_k = recommendations[:k]
    relevant_talks = set(relevant_talks)

    hits = sum(talk_id in relevant_talks for talk_id in top_k)

    precision = hits / k
    recall = hits / len(relevant_talks) if relevant_talks else 0

    return precision, recall

In [18]:
evaluation_rows = []

for row in test_data.itertuples(index=False):
    user_id = row.user_id
    relevant_talk = row.talk_id

    content_list = get_content_recommendations(user_id, top_n=10)
    collab_list = get_collaborative_recommendations(user_id, top_n=10)

    for model_name, recommendations in [
        ("Content", content_list),
        ("Collaborative", collab_list)
    ]:
        precision, recall = precision_recall_at_k(
            recommendations,
            [relevant_talk],
            k=10
        )

        evaluation_rows.append({
            "user_id": user_id,
            "model": model_name,
            "Precision@10": precision,
            "Recall@10": recall
        })

individual_results = pd.DataFrame(evaluation_rows)

summary = individual_results.groupby("model")[["Precision@10", "Recall@10"]].mean().reset_index()

display(summary)

/tmp/ipykernel_2876/1639321295.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores = weighted_scores / similarity_totals.replace(0, np.nan)
/tmp/ipykernel_2876/1639321295.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores = weighted_scores / similarity_totals.replace(0, np.nan)
/tmp/ipykernel_2876/1639321295.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the f

,model,Precision@10,Recall@10
0,Collaborative,0.033333,0.333333
1,Content,0.006667,0.066667


In [19]:
def normalize_scores(scores):
    if scores.empty or scores.max() == scores.min():
        return scores * 0
    return (scores - scores.min()) / (scores.max() - scores.min())


def get_hybrid_recommendations(user_id, content_weight=0.25, top_n=10):
    content_ids = get_content_recommendations(user_id, top_n=50)
    collab_ids = get_collaborative_recommendations(user_id, top_n=50)

    content_scores = pd.Series(
        range(len(content_ids), 0, -1),
        index=content_ids,
        dtype=float
    )
    collab_scores = pd.Series(
        range(len(collab_ids), 0, -1),
        index=collab_ids,
        dtype=float
    )

    content_scores = normalize_scores(content_scores)
    collab_scores = normalize_scores(collab_scores)

    all_ids = set(content_ids) | set(collab_ids)
    rows = []

    for talk_id in all_ids:
        score = (
            content_weight * content_scores.get(talk_id, 0)
            + (1 - content_weight) * collab_scores.get(talk_id, 0)
        )
        rows.append({"talk_id": talk_id, "hybrid_score": score})

    result = pd.DataFrame(rows).sort_values("hybrid_score", ascending=False)
    return result["talk_id"].head(top_n).tolist()


hybrid_rows = []

for weight in [0.0, 0.25, 0.5, 0.75, 1.0]:
    for row in test_data.itertuples(index=False):
        recommendations = get_hybrid_recommendations(
            row.user_id,
            content_weight=weight,
            top_n=10
        )

        precision, recall = precision_recall_at_k(
            recommendations,
            [row.talk_id],
            k=10
        )

        hybrid_rows.append({
            "content_weight": weight,
            "collaborative_weight": 1 - weight,
            "Precision@10": precision,
            "Recall@10": recall
        })

hybrid_results = pd.DataFrame(hybrid_rows)

weight_summary = hybrid_results.groupby(
    ["content_weight", "collaborative_weight"]
)[["Precision@10", "Recall@10"]].mean().reset_index()

display(weight_summary)

/tmp/ipykernel_2876/1639321295.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores = weighted_scores / similarity_totals.replace(0, np.nan)
/tmp/ipykernel_2876/1639321295.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores = weighted_scores / similarity_totals.replace(0, np.nan)
/tmp/ipykernel_2876/1639321295.py:36: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the f

,content_weight,collaborative_weight,Precision@10,Recall@10
0,0.00,1.00,0.033333,0.333333
1,0.25,0.75,0.040000,0.400000
2,0.50,0.50,0.023333,0.233333
3,0.75,0.25,0.013333,0.133333
4,1.00,0.00,0.006667,0.066667


In [20]:
print("Individual model results:")
display(summary)

print("Hybrid weight results:")
display(weight_summary)

best_setting = weight_summary.sort_values(
    ["Recall@10", "Precision@10"],
    ascending=False
).iloc[0]

print(
    "Best tested content weight:",
    best_setting["content_weight"],
    "| collaborative weight:",
    best_setting["collaborative_weight"]
)

Individual model results:


,model,Precision@10,Recall@10
0,Collaborative,0.033333,0.333333
1,Content,0.006667,0.066667


Hybrid weight results:


,content_weight,collaborative_weight,Precision@10,Recall@10
0,0.00,1.00,0.033333,0.333333
1,0.25,0.75,0.040000,0.400000
2,0.50,0.50,0.023333,0.233333
3,0.75,0.25,0.013333,0.133333
4,1.00,0.00,0.006667,0.066667


Best tested content weight: 0.25 | collaborative weight: 0.75
